# FastGen — From a Slow Teacher to a Fast Student (CIFAR-10)

An **end-to-end, educational walkthrough** of *diffusion distillation* with
[FastGen](https://github.com/NVlabs/FastGen). By the end you will have trained a
**one-step** image generator and measured how much faster — and how close in quality — it
is compared to its multi-step teacher.

## The big idea

Modern diffusion models such as **EDM** produce great images, but they are **slow**:
generating one image runs the network many times (≈18 steps for EDM on CIFAR-10), each step
denoising a little more.

**Distillation** trains a faster **student** network to reproduce the teacher's output in
*far fewer* steps — often just **1**. The student learns to "jump straight to the answer"
instead of walking the long denoising path. We use **DMD2** (*Distribution Matching
Distillation v2*), which trains the student with a discriminator so its one-step samples
match the teacher's distribution.

```
Teacher (EDM, ~18 steps)  ──distill──►  Student (DMD2, 1 step)
  high quality, slow                     comparable quality, ~18× fewer network calls
```

## What you'll do
1. **Use the preconfigured container environment** — the notebook is intended to run inside the provided container.
2. **Get data & references** — CIFAR-10 and precomputed **FID** reference statistics.
3. **Train the student** — a short DMD2 distillation run (a *test* config, not full convergence).
4. **Evaluate both models** — compute **FID** for teacher vs student and compare speed.

## How quality is measured: FID
**Fréchet Inception Distance (FID)** compares the feature statistics of generated images to
those of real images — **lower is better**. We download precomputed reference statistics so
FID can be computed without re-processing the whole real dataset each time.

> ⏱️ **Heads-up:** this is a *workshop-sized* run. We train only ~5k steps (full convergence
> is ~100k steps ≈ 12–14 h on 8×H100), so the student's FID here will be **worse than the
> published numbers** — that's expected.

## How this notebook runs shell commands

You'll see two ways to run non-Python code:

| Syntax | Scope | Use it for |
|--------|-------|------------|
| `!command` | runs **one line** in a sub-shell | quick one-liners (`!nvidia-smi`) |
| `%%bash` (must be the **first line** of the cell) | runs the **whole cell** as a Bash script | multi-line setup, `cd`, loops, here-docs |

Shell cells below run directly in the container environment. They do not need any extra conda
setup or kernel registration steps.

> The **Python** cells (training, evaluation) instead run inside the active notebook kernel, so
> they already see the container environment.

## Part 1: Downloading Data and References

The container environment is already prepared, so we can start directly with the CIFAR-10
download and conversion step.

### 1.1 Download the CIFAR-10 Dataset

Downloads CIFAR-10 (60,000 colour images, 32×32, 10 classes) and converts it into the format
FastGen's data loaders expect. This is the data the teacher was trained on and the target
distribution the student must learn to match.

In [ ]:
%%bash
cd /workspace/user_homes/hseth/FastGen
python scripts/download_data.py --dataset cifar10 --max-samples 10000

## Part 2: Downloading FID References and Adding Credentials

The CIFAR-10 data download above has already prepared the training set. We now add the FID
reference statistics and optionally configure W&B credentials for logging.

In [ ]:
# The CIFAR-10 dataset download already ran above.
print("CIFAR-10 download completed above; continuing with FID references and credentials.")

#### Preview the dataset — what are we generating?

Before training, let's look at the **real** data. CIFAR-10 is **50,000** training images at
**32×32** resolution across **10 classes**. The student model's job is to learn to generate
*new* images that look like these. We show one real example per class below (upscaled 4× with
nearest-neighbour, so you see the true — quite low! — resolution).

> Runs in the **notebook kernel** (uses PIL + NumPy, already available in the container). If you see
> a `ModuleNotFoundError`, switch to the active container kernel and re-run the cell.

In [ ]:
import os, io, json, zipfile, glob
import numpy as np
import PIL.Image as Image
import PIL.ImageDraw as ImageDraw
from IPython.display import display

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog',
                   'frog','horse','ship','truck']

# Locate the EDM-format CIFAR-10 zip produced by the download cell above.
candidates = [
    os.path.join(os.environ.get("DATA_ROOT_DIR", "FASTGEN_OUTPUT/DATA"), "cifar10/cifar10-32x32.zip"),
    "FASTGEN_OUTPUT/DATA/cifar10/cifar10-32x32.zip",
] + glob.glob("**/cifar10-32x32.zip", recursive=True)
zip_path = next((p for p in candidates if p and os.path.exists(p)), None)
assert zip_path, "cifar10-32x32.zip not found — run the CIFAR-10 download cell (2.1) first."
print(f"Dataset archive: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    pngs = sorted(n for n in names if n.lower().endswith(".png"))
    labels = {}
    if "dataset.json" in names:
        for fname, lab in (json.loads(z.read("dataset.json")).get("labels") or []):
            labels[fname] = lab
    print(f"Images: {len(pngs):,} | 32x32 RGB | classes: {len(CIFAR10_CLASSES)} | labelled: {len(labels):,}")

    # One example per class (fall back to the first 10 images if labels are missing).
    picks = []
    if labels:
        for cls in range(len(CIFAR10_CLASSES)):
            hit = next((f for f in pngs if labels.get(f) == cls), None)
            if hit:
                picks.append((hit, cls))
    if not picks:
        picks = [(f, labels.get(f)) for f in pngs[:10]]

    # Compose a labelled montage with PIL (no matplotlib needed).
    COLS, SCALE, TILE = 5, 4, 32
    UP, PAD, LBL = TILE * SCALE, 8, 16
    rows = (len(picks) + COLS - 1) // COLS
    cell_w, cell_h = UP + PAD, UP + PAD + LBL
    canvas = Image.new("RGB", (COLS * cell_w + PAD, rows * cell_h + PAD), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    for i, (fname, cls) in enumerate(picks):
        img = Image.open(io.BytesIO(z.read(fname))).convert("RGB").resize((UP, UP), Image.NEAREST)
        r, c = divmod(i, COLS)
        x, y = PAD + c * cell_w, PAD + r * cell_h
        draw.text((x + 2, y), CIFAR10_CLASSES[cls] if cls is not None else "?", fill=(0, 0, 0))
        canvas.paste(img, (x, y + LBL))

print("Real CIFAR-10 training images (one per class):")
display(canvas)

### 2.2 Download CIFAR-10 FID Reference Statistics

FID compares generated images against the **statistics of the real images** (the mean and
covariance of their Inception features). Downloading these precomputed reference statistics
means we don't have to recompute them from scratch on every evaluation.

In [ ]:
%%bash
cd /workspace/user_homes/hseth/FastGen
python scripts/download_data.py --dataset cifar10 --compute-fid-refs

### 2.3 Configure Weights & Biases (W&B) — optional

FastGen logs training metrics to [Weights & Biases](https://wandb.ai). A key is **optional**:

- **Have a key?** Paste it below (get one at https://wandb.ai/authorize) — runs stream to your
  W&B dashboard.
- **No key?** Leave it as an empty string — the cell switches W&B to **offline mode** and
  training still works, logging locally instead of to the cloud.

In [1]:
import os

# Paste your W&B API key between the quotes, or leave empty for offline mode.
token = ''  # e.g. token = 'abcd1234yourkeyhere'

os.makedirs('credentials', exist_ok=True)

with open('credentials/wandb_api.txt', 'w', encoding='utf-8') as f:
    f.write(token)
os.environ['WANDB_API_KEY'] = token
print('✅ W&B key saved to credentials/wandb_api.txt — runs will sync to wandb.ai')


✅ W&B key saved to credentials/wandb_api.txt — runs will sync to wandb.ai


## Part 3: Training the Student Model

### What training actually does

DMD2 distillation runs two networks together:
- the **teacher** (frozen pre-trained EDM) provides the target score/distribution, and
- a **discriminator** pushes the **student**'s one-step samples to be indistinguishable from
  the teacher's, while a distribution-matching loss keeps them aligned.

The result is a student that turns pure noise into a realistic image in a **single forward
pass**.

### Configuration

**Current config:** `config_dmd2_test.py`

- **Method:** DMD2 (*Distribution Matching Distillation v2*) — discriminator-guided distillation
- **Teacher:** pre-trained **EDM** on CIFAR-10
- **Student:** lightweight few-step (here, 1-step) sampler

**Other CIFAR-10 distillation configs you can swap in:**
- `config_cm_cifar10.py` — Consistency Model distillation
- `config_tcm_cifar10.py` — Trajectory Consistency Model
- `config_scd_cifar10.py` — Score-based Consistency Distillation
- `config_sct_cifar10.py` — Sequence Consistency Training
- `config_mf_cifar10.py` — MeanFlow
- `config_cm_cifar10_fast.py` — Fast Consistency Model variant

### 3.1 Train the Student Model

*Note:* In this workshop we run only the test config `config_dmd2_test.py` for ~5k steps.
Full convergence of the DMD2 student takes roughly 100k steps and ~12–14 h on 8×H100. Because
we stop early, the evaluation FID will reflect this **partial** training run.

In [ ]:
%%bash
# Run training in the container environment, where the required Python packages are already available.
# This shell uses the same runtime as the notebook kernel, so no extra conda activation is needed.
export FASTGEN_OUTPUT_ROOT="${FASTGEN_OUTPUT_ROOT:-FASTGEN_OUTPUT}"
CONFIG=fastgen/configs/experiments/EDM/config_dmd2_test.py
LOG_NAME=student_run

# Auto-detect GPU count (the original hardcoded 4, which fails on a 1-GPU box). torchrun
# launches one process per GPU for Distributed Data Parallel (DDP).
NUM_GPUS=$(python -c "import torch; print(torch.cuda.device_count() or 1)")
echo "Detected ${NUM_GPUS} GPU(s). Training student model with log_config.name=${LOG_NAME}"

torchrun --nproc_per_node=${NUM_GPUS} train.py --config=${CONFIG} - trainer.ddp=True log_config.name=${LOG_NAME}

## Part 4: Evaluating Both Models

We compute **FID** (lower = better) for the **teacher** (original EDM) and the **student**
(distilled DMD2), then compare their speed.

### 4.1 Evaluate the Teacher Model (Original EDM)

**Config:** `config_sft_edm_cifar10.py`

This evaluates the original pre-trained EDM with its standard **18 sampling steps** — our
quality reference.

**Why the checkpoint-wrapping step?** FastGen's evaluation scripts expect a checkpoint living
under `FASTGEN_OUTPUT_ROOT/.../checkpoints/` in FastGen's wrapped format
(`{"model": {"net": raw}, "iteration": ...}`). The teacher ships as a plain state dict, so we
load it, wrap it, and save it where the evaluator looks — making the teacher behave like any
other FastGen checkpoint.

> ⚠️ **Re-running?** Delete the old eval folder first:
> `rm -rf $FASTGEN_OUTPUT_ROOT/fastgen/edm_cifar10_sft/EDM_original/`

In [ ]:
import os, sys, json, torch, time

# ── Edit these ───────────────────────────────────────────────────────
CKPT_ROOT_DIR       = os.environ.get("CKPT_ROOT_DIR", "FASTGEN_OUTPUT/MODEL")
FASTGEN_OUTPUT_ROOT = os.environ.get("FASTGEN_OUTPUT_ROOT", "FASTGEN_OUTPUT")
LOG_GROUP           = "edm_cifar10_sft"
LOG_NAME            = "EDM_original"
TEACHER_EVAL_SAMPLES = 500
# ─────────────────────────────────────────────────────────────────────

src_path = f"{CKPT_ROOT_DIR}/cifar10/edm-cifar10-32x32-cond-vp.pth"
dst_dir  = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/checkpoints"
dst_path = f"{dst_dir}/0000001.pth"

# 1. Load raw checkpoint
assert os.path.exists(src_path), f"Checkpoint not found: {src_path}"
raw = torch.load(src_path, map_location="cpu")
print(f"✅ Loaded raw checkpoint from: {src_path}")

# 2. Wrap and save
os.makedirs(dst_dir, exist_ok=True)
torch.save({"model": {"net": raw}, "iteration": 1}, dst_path)
print(f"✅ Wrapped checkpoint saved to: {dst_path}")

# 3. Run FID evaluation
start_time = time.time()
return_code = os.system(
    f"{sys.executable} scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_sft_edm_cifar10.py - eval.num_samples={TEACHER_EVAL_SAMPLES} log_config.name={LOG_NAME} log_config.group={LOG_GROUP}"
)
teacher_eval_time = time.time() - start_time
print(f"\nTeacher evaluation finished in {teacher_eval_time:.2f} seconds")
if return_code != 0:
    raise RuntimeError(f"Teacher evaluation failed with exit code {return_code}")

# 4. Print FID result
fid_path = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/samples/fid.json"
with open(fid_path) as f:
    result = json.load(f)
print("\n📊 FID Results:")
for ckpt, fid in zip(result["ckpt_num"], result["fid"]):
    print(f"   iter {ckpt:>7d} → FID = {fid:.4f}")


### 4.2 Evaluate the Student Model

**Config:** `config_dmd2_cifar10.py`

This evaluates the student trained in Part 3, generating images in just **1 sampling step**.
The cell finds the latest training checkpoint, copies it into the evaluation directory, then
computes FID — so we measure the exact model we just trained.

> ⚠️ **Re-running?** Delete the old eval folder first:
> `rm -rf $FASTGEN_OUTPUT_ROOT/fastgen/evaluation/student_run/`

In [ ]:
import os, sys, json, time, shutil, glob

# ── Edit these ───────────────────────────────────────────────────────
CKPT_ROOT_DIR       = os.environ.get("CKPT_ROOT_DIR", "FASTGEN_OUTPUT/MODEL")
FASTGEN_OUTPUT_ROOT = os.environ.get("FASTGEN_OUTPUT_ROOT", "FASTGEN_OUTPUT")
LOG_GROUP           = "cifar10"
LOG_NAME            = "student_run"
STUDENT_EVAL_SAMPLES = 500
# ─────────────────────────────────────────────────────────────────────

# Step 1: Find the last checkpoint from training
train_ckpt_dir = f"{FASTGEN_OUTPUT_ROOT}/fastgen/{LOG_GROUP}/{LOG_NAME}/checkpoints"
ckpt_files = sorted(glob.glob(os.path.join(train_ckpt_dir, "*.pth")))
assert ckpt_files, f"No checkpoints found in: {train_ckpt_dir}"

last_ckpt = ckpt_files[-1]
print(f"✅ Found last checkpoint: {last_ckpt}")

# Step 2: Copy to evaluation directory with same log_name structure
eval_ckpt_dir = f"{FASTGEN_OUTPUT_ROOT}/fastgen/evaluation/{LOG_NAME}/checkpoints"
os.makedirs(eval_ckpt_dir, exist_ok=True)

eval_ckpt_path = os.path.join(eval_ckpt_dir, os.path.basename(last_ckpt))
shutil.copy2(last_ckpt, eval_ckpt_path)
print(f"✅ Copied checkpoint to: {eval_ckpt_path}")

# Step 3: Run FID evaluation on the copied checkpoint
print(f"\nEvaluating student model: {LOG_NAME}")
start_time = time.time()
return_code = os.system(
    f"{sys.executable} scripts/fid/compute_fid_from_ckpts.py --config fastgen/configs/experiments/EDM/config_dmd2_cifar10.py - eval.num_samples={STUDENT_EVAL_SAMPLES} log_config.name={LOG_NAME} log_config.group=evaluation"
)
student_eval_time = time.time() - start_time
print(f"\nStudent evaluation finished in {student_eval_time:.2f} seconds")
if return_code != 0:
    raise RuntimeError(f"Student evaluation failed with exit code {return_code}")

# Step 4: Print FID results
fid_path = f"{FASTGEN_OUTPUT_ROOT}/fastgen/evaluation/{LOG_NAME}/samples/fid.json"
with open(fid_path) as f:
    result = json.load(f)
print("\n📊 FID Results:")
for ckpt, fid in zip(result["ckpt_num"], result["fid"]):
    print(f"   iter {ckpt:>7d} → FID = {fid:.4f}")

### 4.3 Performance Comparison

Compare wall-clock evaluation time and the resulting speed-up. The student should be
dramatically faster because it uses ~1 step versus the teacher's ~18.

In [ ]:
try:
    print("⏱️  Evaluation Timing Comparison:")
    print(f"   Teacher evaluation: {teacher_eval_time:.2f} seconds")
    print(f"   Student evaluation: {student_eval_time:.2f} seconds")
    speedup = teacher_eval_time / student_eval_time if student_eval_time > 0 else float('inf')
    print(f"   Speedup ratio (Teacher/Student): {speedup:.2f}x")
except NameError as e:
    print('Timing variables not found. Make sure both evaluation cells have been run.')

### 4.4 Visual comparison — teacher vs student samples

FID is abstract — let's actually *look* at what each model generates. We sample a fresh **8×8
grid** from each model using FastGen's sampler (`eval.save_images=True`, which saves a grid and
skips FID), then show them side by side:

- **Teacher (EDM)** runs the full **18-step** sampler.
- **Student (DMD2)** generates in a **single step**.

This reuses the same checkpoints as the eval cells but writes to isolated `gen/` run dirs, so it
never clobbers your FID results. Expect ~1–2 min (model load + sampling for both).

> 🎓 **What you'll see:** if the student was only trained briefly (e.g. the 20-iter smoke test),
> its samples look like **blurry colour blobs** — it hasn't learned the data yet, which is the
> honest, expected result. After full convergence (~100k steps) the 1-step student's grid
> approaches the teacher's — that's the payoff of distillation: near-teacher images in **1
> network call instead of 18**.

In [ ]:
import os, sys, glob, shutil
import torch
import PIL.Image as Image, PIL.ImageDraw as ImageDraw
from IPython.display import display

ROOT = os.environ.get("FASTGEN_OUTPUT_ROOT", "FASTGEN_OUTPUT")
PY = sys.executable
IMAGES_PER_MODEL = 64  # 8x8 grid

TEACHER_CFG = "fastgen/configs/experiments/EDM/config_sft_edm_cifar10.py"
STUDENT_CFG = "fastgen/configs/experiments/EDM/config_dmd2_cifar10.py"

def _place(dst_dir, basename, src=None, wrap_raw=None):
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, basename)
    if not os.path.exists(dst):
        if wrap_raw:                       # wrap a raw EDM state-dict into FastGen format
            raw = torch.load(wrap_raw, map_location="cpu")
            torch.save({"model": {"net": raw}, "iteration": int(basename.split('.')[0])}, dst)
        else:
            shutil.copy2(src, dst)
    return dst

def _generate(cfg, name):
    # Isolated gen/<name> dirs have no fid.json, so the script regenerates instead of skipping.
    cmd = (f"{PY} scripts/fid/compute_fid_from_ckpts.py --config {cfg} "
           f"- eval.save_images=True eval.num_samples={IMAGES_PER_MODEL} "
           f"dataloader_train.batch_size={IMAGES_PER_MODEL} log_config.group=gen log_config.name={name}")
    print(f"Generating {IMAGES_PER_MODEL} samples from '{name}' ...")
    assert os.system(cmd) == 0, f"generation failed for {name}"
    grids = sorted(glob.glob(f"{ROOT}/fastgen/gen/{name}/samples/iter_*/vis/*.png"), key=os.path.getmtime)
    assert grids, f"no grid produced for {name}"
    return grids[-1]

# Teacher: wrap the raw EDM checkpoint; Student: copy the latest training checkpoint.
_place(f"{ROOT}/fastgen/gen/teacher/checkpoints", "0000001.pth",
       wrap_raw=f"{ROOT}/MODEL/cifar10/edm-cifar10-32x32-cond-vp.pth")
train_ckpts = sorted(glob.glob(f"{ROOT}/fastgen/cifar10/student_run/checkpoints/*.pth"))
assert train_ckpts, "No student checkpoint found - run Part 3 (training) first."
_place(f"{ROOT}/fastgen/gen/student/checkpoints", os.path.basename(train_ckpts[-1]), src=train_ckpts[-1])

teacher_grid = _generate(TEACHER_CFG, "teacher")
student_grid = _generate(STUDENT_CFG, "student")

def _labelled(path, title):
    im = Image.open(path).convert("RGB")
    if im.width < 256:
        f = max(1, 256 // im.width)
        im = im.resize((im.width * f, im.height * f), Image.NEAREST)
    out = Image.new("RGB", (im.width, im.height + 22), (255, 255, 255))
    ImageDraw.Draw(out).text((4, 5), title, fill=(0, 0, 0))
    out.paste(im, (0, 22))
    return out

t = _labelled(teacher_grid, "TEACHER  -  EDM  (18 steps)")
s = _labelled(student_grid, "STUDENT  -  DMD2  (1 step)")
GAP = 18
combo = Image.new("RGB", (t.width + GAP + s.width, max(t.height, s.height)), (255, 255, 255))
combo.paste(t, (0, 0)); combo.paste(s, (t.width + GAP, 0))
print("Same pipeline, same noise -> 18 network calls vs 1:")
display(combo)

## Summary & next steps

You just ran the full distillation loop:
- ✅ Built a reproducible `fastgen` environment and verified the GPU.
- ✅ Prepared CIFAR-10 and its FID reference statistics.
- ✅ Distilled a **1-step DMD2 student** from an **18-step EDM teacher** (short test run).
- ✅ Compared **FID** (quality) and **wall-clock time** (speed).

**Key takeaway:** distillation trades a small amount of quality for a large speed-up — the
student approximates many teacher steps in one.

**What to try next**
- Train to convergence: swap `config_dmd2_test.py` → `config_dmd2_cifar10.py` (~100k steps).
- Try another method from the Part 3 table (Consistency Models, MeanFlow, …).
- Raise `*_EVAL_SAMPLES` (e.g. 50000) for a statistically meaningful FID.
- Inspect generated images under `FASTGEN_OUTPUT/fastgen/.../samples/`.

## Documentation

Detailed documentation is available in each component's README:

| Component | Documentation | Description |
|-----------|---------------|-------------|
| **Methods** | [fastgen/methods/README.md](fastgen/methods/README.md) | Training methods (sCM, MeanFlow, DMD2, Self-Forcing, etc.) |
| **Networks** | [fastgen/networks/README.md](fastgen/networks/README.md) | Network architectures (EDM, SD, SDXL, Flux, Qwen-Image, WAN, CogVideoX, Cosmos) and pretrained models |
| **Configs** | [fastgen/configs/README.md](fastgen/configs/README.md) | Configuration system, environment variables, and creating custom configs |
| **Datasets** | [fastgen/datasets/README.md](fastgen/datasets/README.md) | Dataset preparation and WebDataset loaders |
| **Callbacks** | [fastgen/callbacks/README.md](fastgen/callbacks/README.md) | Training callbacks (EMA, logging, gradient clipping, etc.) |
| **Inference** | [scripts/README.md](scripts/README.md) | Inference modes (T2I, T2V, I2V, V2V, etc.) and FID evaluation |
| **Third Party** | [fastgen/third_party/README.md](fastgen/third_party/README.md) | Third-party dependencies (Depth Anything V2, etc.) |